# Pipeline de Padronização de Produtos — MeLi Data Challenge 2019

**Arquitetura em 4 passos:**
1. Ingestão e Amostragem (Micro Gold Standard)
2. Extração via LLM (Camada Semântica)
3. Limpeza Determinística com Python (Camada Simbólica)
4. Exportação para Avaliação Manual


In [ ]:
!pip install tqdm 

In [1]:
import os
from dotenv import load_dotenv
import re
import json
import pandas as pd
from tqdm import tqdm
import google.generativeai as genai
import logging
import time

c:\Users\nunes\.pyenv\pyenv-win\versions\3.12.3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

In [3]:
DATASET_PATH = "data/train.csv"
OUTPUT_PATH = "output/resultado_pipeline.csv"
SAMPLE_SIZE = 200
RANDOM_SEED = 42
API_KEY = os.getenv("API_KEY")   
LLM_MODEL = "gemini-3.1-flash-lite-preview"      
TEMPERATURE = 0.0

## Passo 1 — Ingestão e Amostragem

In [4]:
dataframe = pd.read_csv("data/train.csv")

In [5]:
dataframe.head(5)

,title,label_quality,language,category
0,Hidrolavadora Lavor One 120 Bar 1700w Bomba A...,unreliable,spanish,ELECTRIC_PRESSURE_WASHERS
1,Placa De Sonido - Behringer Umc22,unreliable,spanish,SOUND_CARDS
2,Maquina De Lavar Electrolux 12 Kilos,unreliable,portuguese,WASHING_MACHINES
3,Par Disco De Freio Diant Vent Gol 8v 08/ Frema...,unreliable,portuguese,VEHICLE_BRAKE_DISCS
4,Flashes Led Pestañas Luminoso Falso Pestañas P...,unreliable,spanish,FALSE_EYELASHES


In [6]:
df = dataframe.copy()

In [7]:
df = df[df["language"] == "portuguese"]


In [8]:
df

,title,label_quality,language,category
2,Maquina De Lavar Electrolux 12 Kilos,unreliable,portuguese,WASHING_MACHINES
3,Par Disco De Freio Diant Vent Gol 8v 08/ Frema...,unreliable,portuguese,VEHICLE_BRAKE_DISCS
5,"4 Microaspersor Irrigação Ultra 7,20 Metros",unreliable,portuguese,IRRIGATION_SPRINKLERS
6,Raquete Clash 100 Tour - Nova,unreliable,portuguese,RACQUETS
7,"Kit Tripe Para Celular Ou Câmera Fotog 1,20m +...",unreliable,portuguese,CAMERA_TRIPODS
...,...,...,...,...
19999988,Kit 10 Capas De Almofadas 30x40 Lol,unreliable,portuguese,CUSHION_COVERS
19999991,Conjunto Conexões E Peças Para Osmose Reversa,unreliable,portuguese,AQUARIUM_FILTERS
19999992,"Rodapé Santa Luzia 566 Branco 15cm 2,40 Metros",unreliable,portuguese,BASEBOARDS
19999997,Bateria Portátil 3300 Mah Power Bank Usb Max...,unreliable,portuguese,PORTABLE_CELLPHONE_CHARGERS


In [9]:
df_sample = df.sample(n=200, random_state=42).reset_index(drop=True)

In [10]:
df_sample

,title,label_quality,language,category
0,2 Unidades Caixa Transporte Para Cães E Gatos ...,unreliable,portuguese,DOG_CARRIERS_AND_CARRYING_BAGS
1,Ferro De Passar Dimar Elétrico Funcionando Rar...,unreliable,portuguese,IRONS
2,Bateria Carregador Celular Samsung Young 2 130,unreliable,portuguese,CELLPHONE_BATTERIES
3,Vassoura Elétrica Portátil Aspirador Pó Vermel...,unreliable,portuguese,ELECTRIC_BROOMS
4,Depilador Tipo Laser Yes Finishing Touch Envio...,unreliable,portuguese,EPILATORS
...,...,...,...,...
195,Espátula Perfurada De Inox Com Nylon - Brinox ...,unreliable,portuguese,KITCHEN_SHOVELS_AND_SPATULAS
196,Nicho Decorativo Conjunto 2 Caixas Unicas Em M...,unreliable,portuguese,HOME_SHELVES
197,Kit Ferreiraczarntraíra Perfil Baixo + Vara ...,unreliable,portuguese,FISHING_RODS
198,9 Cuecas Boxer Grafite Microfibra S/ Costura L...,unreliable,portuguese,MALE_UNDERWEAR


## Passo 2 — Extração via LLM (Camada Semântica)

In [11]:
SYSTEM_PROMPT = """
Você é um especialista em extração de atributos de produtos de e-commerce brasileiro.

Sua tarefa:
Receba o TÍTULO BRUTO de um produto e retorne SOMENTE um objeto JSON válido,
sem texto adicional, sem markdown, sem blocos de código.

Chaves obrigatórias no JSON (use null se o atributo não existir no título):
- "marca"              : string — ex: "Electrolux", "Samsung", "Tramontina"
- "modelo"             : string — ex: "LTD12", "Galaxy S21", null
- "tipo_produto"       : string — ex: "máquina de lavar", "geladeira", "liquidificador"
- "capacidade_volume"  : string — ex: "12kg", "500ml", "2l", null
- "voltagem"           : string — SEMPRE no formato "110v", "220v" ou "bivolt"; null se ausente
- "cor"                : string — ex: "branco", "preto", null
- "material"           : string — ex: "inox", "plástico", null
- "quantidade"         : string — ex: "kit 3 peças", "par", null
- "outros_atributos"   : string — resumo de outros atributos relevantes, null se não houver

REGRAS CRÍTICAS DE PRÉ-NORMALIZAÇÃO:

1. Conversão de volume:
   - "1/2 litro", "meio litro", "0,5 litro" → "500ml"
   - "1 litro", "1l"                         → "1000ml"
   - "1,5 litro", "um litro e meio"          → "1500ml"
   - "2 litros"                              → "2000ml"
   - Sempre: número + "ml" sem espaço

2. Padronização de voltagem:
   - "110 v", "110 volts", "110volts", "110V" → "110v"
   - "220 v", "220 volts", "220V"             → "220v"
   - "bivolt", "bi-volt", "Bi Volt"           → "bivolt"

3. Padronização de peso/capacidade:
   - "12 kilos", "12 kg", "12KG"             → "12kg"
   - "500 g", "500gr", "500 gramas"          → "500g"

4. Texto em minúsculo para todos os valores, exceto nomes de marcas (capitalize a 1ª letra).

EXEMPLOS (Few-Shot):

Título: "Maquina De Lavar Electrolux 12 Kilos"
JSON:
{\"marca\":\"Electrolux\",\"modelo\":null,\"tipo_produto\":\"máquina de lavar\",\"capacidade_volume\":\"12kg\",\"voltagem\":null,\"cor\":null,\"material\":null,\"quantidade\":null,\"outros_atributos\":null}

Título: "Liquidificador Mondial Turbo Inox 1000w 110v 3 Velocidades"
JSON:
{\"marca\":\"Mondial\",\"modelo\":\"Turbo Inox\",\"tipo_produto\":\"liquidificador\",\"capacidade_volume\":null,\"voltagem\":\"110v\",\"cor\":null,\"material\":\"inox\",\"quantidade\":null,\"outros_atributos\":\"1000w, 3 velocidades\"}

Título: "Garrafa Térmica 1/2 Litro Aço Inox Com Alça"
JSON:
{\"marca\":null,\"modelo\":null,\"tipo_produto\":\"garrafa térmica\",\"capacidade_volume\":\"500ml\",\"voltagem\":null,\"cor\":null,\"material\":\"aço inox\",\"quantidade\":null,\"outros_atributos\":\"com alça\"}
"""

In [12]:
def extract_attributes_llm(title: str, client: genai.GenerativeModel) -> dict:
    try:
        response = client.generate_content(f'Título: "{title}"')
        raw_text = response.text.strip()
        
        # Limpa markdown se houver
        if raw_text.startswith("```"):
            raw_text = re.sub(r"^```(?:json)?", "", raw_text).rstrip("`").strip()

        # Tenta JSON direto
        if raw_text:
            try:
                return json.loads(raw_text)
            except json.JSONDecodeError:
                pass

        # Fallback: extrai o primeiro {...} encontrado no texto
        match = re.search(r"\{.*\}", raw_text, re.DOTALL)
        if match:
            return json.loads(match.group())

        logger.warning("Sem JSON extraível para '%s'. Texto: %s", title, repr(raw_text))
        return {}

    except ValueError as e:
        logger.warning("Resposta bloqueada para '%s': %s", title, e)
        return {}
    except Exception as e:
        logger.error("Erro LLM para '%s': %s", title, e)
        return {}

## Passo 3 — Limpeza Determinística (Camada Simbólica)

In [13]:
_RE_SPACES = re.compile(r"\s+")
_RE_VOLUME_ML = re.compile(r"(\d+(?:[.,]\d+)?)\s*ml", re.IGNORECASE)
_RE_VOLUME_L = re.compile(r"(\d+(?:[.,]\d+)?)\s*l(?:itro[s]?)?(?=\b|$)", re.IGNORECASE)
_RE_WEIGHT_KG = re.compile(r"(\d+(?:[.,]\d+)?)\s*k(?:g|ilos?|ilogramas?)", re.IGNORECASE)
_RE_WEIGHT_G = re.compile(r"(\d+(?:[.,]\d+)?)\s*g(?:r|ramas?)?(?=\b|$)", re.IGNORECASE)
_RE_VOLTAGE = re.compile(r"(\d+)\s*v(?:olts?)?(?=\b|$)", re.IGNORECASE)
_RE_BIVOLT = re.compile(r"bi[\s-]?volt", re.IGNORECASE)

In [14]:
def _clean_str(value: str | None) -> str:
    """Remove espaços excedentes e converte para minúsculo."""
    if not value or not isinstance(value, str):
        return ""
    return _RE_SPACES.sub(" ", value.strip()).lower()

In [15]:
def _normalize_volume(value: str) -> str:
    """Converte volume para formato canônico em ml."""
    if not value:
        return value
    value = value.replace(",", ".")
    m = _RE_VOLUME_ML.search(value)
    if m:
        qty = float(m.group(1))
        return f"{int(qty)}ml" if qty == int(qty) else f"{qty}ml"
    m = _RE_VOLUME_L.search(value)
    if m:
        return f"{int(float(m.group(1)) * 1000)}ml"
    return value


In [16]:
def _normalize_weight(value: str) -> str:
    """Padroniza peso para 'Xkg' ou 'Xg'."""
    if not value:
        return value
    value = value.replace(",", ".")
    m = _RE_WEIGHT_KG.search(value)
    if m:
        qty = float(m.group(1))
        return f"{int(qty)}kg" if qty == int(qty) else f"{qty}kg"
    m = _RE_WEIGHT_G.search(value)
    if m:
        qty = float(m.group(1))
        return f"{int(qty)}g" if qty == int(qty) else f"{qty}g"
    return value

In [17]:
def _normalize_voltage(value: str) -> str:
    """Padroniza voltagem para '110v', '220v' ou 'bivolt'."""
    if not value:
        return value
    if _RE_BIVOLT.search(value):
        return "bivolt"
    m = _RE_VOLTAGE.search(value)
    return f"{m.group(1)}v" if m else value

In [18]:
def _normalize_capacity(value: str) -> str:
    """Detecta se é volume ou peso e aplica normalização correta."""
    if not value:
        return value
    v = value.lower()
    if any(u in v for u in ["ml", "litro", " l", "l "]):
        return _normalize_volume(v)
    if any(u in v for u in ["kg", "kilo", "grama", " g", "g "]):
        return _normalize_weight(v)
    return _clean_str(value)

In [19]:
def clean_llm_output(raw_json: dict) -> dict:
    """
    Higieniza o JSON bruto do LLM:
    - Minúsculo + sem espaços extras
    - Re-normaliza voltagem, volume e peso via Regex
    - Marca: title-case
    """
    if not raw_json:
        return {}
    cleaned = {}
    for key, value in raw_json.items():
        if value is None or value == "null":
            cleaned[key] = None; continue
        s = str(value).strip()
        if key == "marca":
            cleaned[key] = _RE_SPACES.sub(" ", s).title() or None
        elif key == "voltagem":
            cleaned[key] = _normalize_voltage(_clean_str(s)) or None
        elif key == "capacidade_volume":
            cleaned[key] = _normalize_capacity(_clean_str(s)) or None
        else:
            cleaned[key] = _clean_str(s) or None
    return cleaned

In [20]:
def build_canonical_name(cleaned_json: dict) -> str:
    """
    Concatena atributos higienizados na ordem canônica:
    tipo_produto > marca > modelo > capacidade > voltagem > cor > material > qtd > outros
    """
    ORDER = ["tipo_produto","marca","modelo","capacidade_volume",
             "voltagem","cor","material","quantidade","outros_atributos"]
    parts = [str(cleaned_json[f]) for f in ORDER
             if cleaned_json.get(f) not in (None, "", "null")]
    return " | ".join(parts)

## Passo 4 — Orquestração e Exportação

In [21]:
os.makedirs("output", exist_ok=True)

In [22]:
genai.configure(api_key=API_KEY)
client = genai.GenerativeModel(
    model_name=LLM_MODEL,
    system_instruction=SYSTEM_PROMPT,
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",
        temperature=TEMPERATURE,
        max_output_tokens=512,
    )
)

In [23]:
raw_jsons, cleaned_jsons, canonical_names = [], [], []

for title in tqdm(df_sample["title"], desc="Processando títulos", unit="item"):
    raw     = extract_attributes_llm(title, client)
    cleaned = clean_llm_output(raw)
    canon   = build_canonical_name(cleaned)

    raw_jsons.append(raw)
    cleaned_jsons.append(cleaned)
    canonical_names.append(canon)
    time.sleep(5)

Processando títulos:   0%|          | 0/200 [00:00<?, ?item/s]

Processando títulos: 100%|██████████| 200/200 [52:59<00:00, 15.90s/item]  


In [24]:
df_result = pd.DataFrame({
    "titulo_original"   : df_sample["title"].values,
    "categoria_original": df_sample["category"].values,
    "json_llm_bruto"    : [json.dumps(r, ensure_ascii=False) for r in raw_jsons],
    "json_limpo_python" : [json.dumps(c, ensure_ascii=False) for c in cleaned_jsons],
    "nome_canonico_final": canonical_names,
})

In [25]:
df_result

,titulo_original,categoria_original,json_llm_bruto,json_limpo_python,nome_canonico_final
0,2 Unidades Caixa Transporte Para Cães E Gatos ...,DOG_CARRIERS_AND_CARRYING_BAGS,"{""marca"": ""Pet Grillo"", ""modelo"": ""n°2"", ""tipo...","{""marca"": ""Pet Grillo"", ""modelo"": ""n°2"", ""tipo...",caixa de transporte | Pet Grillo | n°2 | 2 uni...
1,Ferro De Passar Dimar Elétrico Funcionando Rar...,IRONS,"{""marca"": ""Dimar"", ""modelo"": null, ""tipo_produ...","{""marca"": ""Dimar"", ""modelo"": null, ""tipo_produ...","ferro de passar | Dimar | elétrico, funcionand..."
2,Bateria Carregador Celular Samsung Young 2 130,CELLPHONE_BATTERIES,"{""marca"": ""Samsung"", ""modelo"": ""Young 2 130"", ...","{""marca"": ""Samsung"", ""modelo"": ""young 2 130"", ...",bateria | Samsung | young 2 130 | carregador c...
3,Vassoura Elétrica Portátil Aspirador Pó Vermel...,ELECTRIC_BROOMS,"{""marca"": ""Philco"", ""modelo"": null, ""tipo_prod...","{""marca"": ""Philco"", ""modelo"": null, ""tipo_prod...",vassoura elétrica | Philco | 220v | vermelho |...
4,Depilador Tipo Laser Yes Finishing Touch Envio...,EPILATORS,"{""marca"": ""Yes Finishing Touch"", ""modelo"": nul...","{""marca"": ""Yes Finishing Touch"", ""modelo"": nul...","depilador | Yes Finishing Touch | tipo laser, ..."
...,...,...,...,...,...
195,Espátula Perfurada De Inox Com Nylon - Brinox ...,KITCHEN_SHOVELS_AND_SPATULAS,"{""marca"": ""Brinox"", ""modelo"": ""5977"", ""tipo_pr...","{""marca"": ""Brinox"", ""modelo"": ""5977"", ""tipo_pr...",espátula perfurada | Brinox | 5977 | inox com ...
196,Nicho Decorativo Conjunto 2 Caixas Unicas Em M...,HOME_SHELVES,"{""marca"": null, ""modelo"": ""Stucco"", ""tipo_prod...","{""marca"": null, ""modelo"": ""stucco"", ""tipo_prod...",nicho decorativo | stucco | mdf | conjunto 2 p...
197,Kit Ferreiraczarntraíra Perfil Baixo + Vara ...,FISHING_RODS,"{""marca"": ""Ferreiraczarntraíra"", ""modelo"": nul...","{""marca"": ""Ferreiraczarntraíra"", ""modelo"": nul...",kit de pesca | Ferreiraczarntraíra | carbono |...
198,9 Cuecas Boxer Grafite Microfibra S/ Costura L...,MALE_UNDERWEAR,"{""marca"": ""Lupo"", ""modelo"": ""436001"", ""tipo_pr...","{""marca"": ""Lupo"", ""modelo"": ""436001"", ""tipo_pr...",cueca boxer | Lupo | 436001 | grafite | microf...


In [26]:
df_result.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Exportado para: {OUTPUT_PATH}")
df_result.head(10)

Exportado para: output/resultado_pipeline.csv


,titulo_original,categoria_original,json_llm_bruto,json_limpo_python,nome_canonico_final
0,2 Unidades Caixa Transporte Para Cães E Gatos ...,DOG_CARRIERS_AND_CARRYING_BAGS,"{""marca"": ""Pet Grillo"", ""modelo"": ""n°2"", ""tipo...","{""marca"": ""Pet Grillo"", ""modelo"": ""n°2"", ""tipo...",caixa de transporte | Pet Grillo | n°2 | 2 uni...
1,Ferro De Passar Dimar Elétrico Funcionando Rar...,IRONS,"{""marca"": ""Dimar"", ""modelo"": null, ""tipo_produ...","{""marca"": ""Dimar"", ""modelo"": null, ""tipo_produ...","ferro de passar | Dimar | elétrico, funcionand..."
2,Bateria Carregador Celular Samsung Young 2 130,CELLPHONE_BATTERIES,"{""marca"": ""Samsung"", ""modelo"": ""Young 2 130"", ...","{""marca"": ""Samsung"", ""modelo"": ""young 2 130"", ...",bateria | Samsung | young 2 130 | carregador c...
3,Vassoura Elétrica Portátil Aspirador Pó Vermel...,ELECTRIC_BROOMS,"{""marca"": ""Philco"", ""modelo"": null, ""tipo_prod...","{""marca"": ""Philco"", ""modelo"": null, ""tipo_prod...",vassoura elétrica | Philco | 220v | vermelho |...
4,Depilador Tipo Laser Yes Finishing Touch Envio...,EPILATORS,"{""marca"": ""Yes Finishing Touch"", ""modelo"": nul...","{""marca"": ""Yes Finishing Touch"", ""modelo"": nul...","depilador | Yes Finishing Touch | tipo laser, ..."
5,Coxim Diant Motor L200 4x4 95/ Mobensani ...,AUTOMOTIVE_ENGINE_MOUNTS,"{""marca"": ""Mobensani"", ""modelo"": ""L200 4x4"", ""...","{""marca"": ""Mobensani"", ""modelo"": ""l200 4x4"", ""...",coxim dianteiro de motor | Mobensani | l200 4x...
6,Álbum De Fotos Floral De Couro Ecol.200 Fotos ...,PHOTO_ALBUMS,"{""marca"": null, ""modelo"": null, ""tipo_produto""...","{""marca"": null, ""modelo"": null, ""tipo_produto""...",álbum de fotos | 200 fotos | preto | couro eco...
7,Jaqueta Couro Personalizada Motociclista C/ Pr...,MOTORCYCLE_JACKETS,"{""marca"": null, ""modelo"": null, ""tipo_produto""...","{""marca"": null, ""modelo"": null, ""tipo_produto""...","jaqueta | couro | personalizada, motociclista,..."
8,Café Poesia Torrado Em Grão 1 Kg .............,GROUND_AND_WHOLE_BEANS_COFFEE,"{""marca"": ""Poesia"", ""modelo"": null, ""tipo_prod...","{""marca"": ""Poesia"", ""modelo"": null, ""tipo_prod...","café | Poesia | 1kg | torrado em grão, exportação"
9,Tampa Traseira Passat 2010,AUTOMOTIVE_TRUNK_LIDS,"{""marca"": ""Volkswagen"", ""modelo"": ""Passat 2010...","{""marca"": ""Volkswagen"", ""modelo"": ""passat 2010...",tampa traseira | Volkswagen | passat 2010 | an...


## Verificação Rápida — Exemplos do Pipeline

In [ ]:
test_titles_1 = [
    "COCA COLA 500 ml",
    "coca-cola 500ml",
    "Coca Cola 0,5 litro",
    "CocaCola 1/2 litro",
    "Coca Cola meio litro",
    "Coca-Cola 500ML",
    "Coca Cola 0.5L",
]

for title in test_titles_1:
    raw     = extract_attributes_llm(title, client)
    cleaned = clean_llm_output(raw)
    canon   = build_canonical_name(cleaned)
    print(f"Título: {title}")
    print(f"Extraído: {raw}")
    print(f"Limpo: {cleaned}")
    print(f"Canônico: {canon}")
    print()

Título: COCA COLA 500 ml
Extraído: {'marca': 'Coca cola', 'modelo': None, 'tipo_produto': 'refrigerante', 'capacidade_volume': '500ml', 'voltagem': None, 'cor': None, 'material': None, 'quantidade': None, 'outros_atributos': None}
Limpo: {'marca': 'Coca Cola', 'modelo': None, 'tipo_produto': 'refrigerante', 'capacidade_volume': '500ml', 'voltagem': None, 'cor': None, 'material': None, 'quantidade': None, 'outros_atributos': None}
Canônico: refrigerante | Coca Cola | 500ml

Título: coca-cola 500ml
Extraído: {'marca': 'Coca-cola', 'modelo': None, 'tipo_produto': 'refrigerante', 'capacidade_volume': '500ml', 'voltagem': None, 'cor': None, 'material': None, 'quantidade': None, 'outros_atributos': None}
Limpo: {'marca': 'Coca-Cola', 'modelo': None, 'tipo_produto': 'refrigerante', 'capacidade_volume': '500ml', 'voltagem': None, 'cor': None, 'material': None, 'quantidade': None, 'outros_atributos': None}
Canônico: refrigerante | Coca-Cola | 500ml

Título: Coca Cola 0,5 litro
Extraído: {'marca

In [34]:
for title in test_titles_1:
    print(f"Título: {title}")
    print(f"Canônico: {canon}")
    print()

Título: COCA COLA 500 ml
Canônico: refrigerante | Coca Cola | 500ml

Título: coca-cola 500ml
Canônico: refrigerante | Coca Cola | 500ml

Título: Coca Cola 0,5 litro
Canônico: refrigerante | Coca Cola | 500ml

Título: CocaCola 1/2 litro
Canônico: refrigerante | Coca Cola | 500ml

Título: Coca Cola meio litro
Canônico: refrigerante | Coca Cola | 500ml

Título: Coca-Cola 500ML
Canônico: refrigerante | Coca Cola | 500ml

Título: Coca Cola 0.5L
Canônico: refrigerante | Coca Cola | 500ml



In [35]:
test_titles_2 = [
    "COCA COLA 2000 ml",
    "coca-cola 500ml",
    "Coca Cola 0,5 litro",
    "CocaCola  2 litro",
    "Coca Cola meio litro",
    "Coca-Cola 1 litro",
    "Coca Cola 0.5L",
    "Coca Cola 1/2L",
]

for title in test_titles_2:
    raw     = extract_attributes_llm(title, client)
    cleaned = clean_llm_output(raw)
    canon   = build_canonical_name(cleaned)
    print(f"Título: {title}")
    print(f"Canônico: {canon}")
    print()

Título: COCA COLA 2000 ml
Canônico: refrigerante | Coca Cola | 2000ml

Título: coca-cola 500ml
Canônico: refrigerante | Coca-Cola | 500ml

Título: Coca Cola 0,5 litro
Canônico: refrigerante | Coca Cola | 500ml

Título: CocaCola  2 litro
Canônico: refrigerante | Cocacola | 2000ml

Título: Coca Cola meio litro
Canônico: refrigerante | Coca Cola | 500ml

Título: Coca-Cola 1 litro
Canônico: refrigerante | Coca-Cola | 1000ml

Título: Coca Cola 0.5L
Canônico: refrigerante | Coca Cola | 500ml

Título: Coca Cola 1/2L
Canônico: refrigerante | Coca Cola | 500ml

